# <font color="steelblue">05. Tablas de frecuencias: contar bien antes de publicar</font>

*Unidad 2. Estadística descriptiva* · Sesión teórica 5 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv"]
PREGUNTAS = ["q_P05_1.json", "q_P05_2.json", "q_P05_3.json", "q_P05_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«El 22 % de los encuestados se informa principalmente por redes sociales.» Detrás de esa frase hay una **tabla de frecuencias**: cuántas personas eligen cada opción y qué parte del total suponen. Es el resumen más sencillo y el más publicado.

**Qué necesitas saber antes**: el cuaderno 4 (leer y filtrar tablas).

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Construir una tabla de frecuencias absolutas y relativas.
2. Calcular frecuencias acumuladas en variables ordinales.
3. Agrupar una variable cuantitativa en intervalos.
4. Redactar porcentajes sin errores de base.

## <font color='steelblue'>Contenidos</font>

1. Frecuencias absolutas y relativas
2. Frecuencias acumuladas en variables ordinales
3. Agrupar en intervalos
4. Porcentajes sin trampas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Frecuencias absolutas y relativas</font>

In [ ]:
encuesta = pd.read_csv("encuesta_medios.csv")

- **Frecuencia absoluta** ($n_i$): cuántas veces aparece cada valor.
- **Frecuencia relativa** ($f_i = n_i / n$): qué proporción del total supone. Multiplicada por 100, es el **porcentaje**.

Las relativas siempre suman 1 (o 100 %).

In [ ]:
absoluta = encuesta["medio_principal"].value_counts()
relativa = encuesta["medio_principal"].value_counts(normalize=True)

tabla = pd.DataFrame({"personas": absoluta, "porcentaje": (relativa * 100).round(1)})
tabla

**Qué ha pasado**: con `normalize=True`, `value_counts()` divide cada recuento por el total. Hemos juntado las dos columnas en una tabla con `pd.DataFrame()`.

In [ ]:
print("Suma de porcentajes:", round(tabla["porcentaje"].sum(), 1))

Por el redondeo, la suma puede dar 99,9 o 100,1. Es normal; si publicas la tabla, añade una nota: «los porcentajes pueden no sumar 100 por redondeo».

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P05_1.json")

## <font color='steelblue'>2. Frecuencias acumuladas en variables ordinales</font>

Cuando las categorías tienen orden (estudios, nivel de confianza), tiene sentido **acumular**: «el 45 % tiene, como mucho, estudios secundarios». Hay que ordenar las categorías antes con `reindex()`.

In [ ]:
orden = ["Primarios", "Secundarios", "FP", "Universitarios"]
fa = encuesta["estudios"].value_counts().reindex(orden)
tabla_est = pd.DataFrame({"n": fa, "%": (fa / fa.sum() * 100).round(1)})
tabla_est["% acumulado"] = tabla_est["%"].cumsum()
tabla_est

**Cómo leerlo**: la última fila de la columna acumulada es siempre 100 %. La frase «el X % no tiene estudios universitarios» se lee en la fila de FP.

La confianza en los medios (0 a 10) también es ordinal. Ordenamos por el valor con `sort_index()`:

In [ ]:
conf = encuesta["confianza_medios"].value_counts().sort_index()
pd.DataFrame({"n": conf, "% acumulado": (conf.cumsum() / conf.sum() * 100).round(1)})

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P05_2.json")

## <font color='steelblue'>3. Agrupar en intervalos</font>

Con variables cuantitativas que toman muchos valores (la edad) una tabla valor a valor no se lee. Se agrupan en **intervalos** con `pd.cut()`. Los límites los decides tú: usa los habituales en tu fuente (el INE suele usar grupos de edad de 5 o 10 años).

In [ ]:
limites = [17, 29, 44, 64, 100]
etiquetas = ["18-29", "30-44", "45-64", "65 y más"]
encuesta["grupo_edad"] = pd.cut(encuesta["edad"], bins=limites, labels=etiquetas)
encuesta["grupo_edad"].value_counts(sort=False)

**Qué ha pasado**: `pd.cut()` asigna cada edad a su intervalo. Por defecto el límite derecho se incluye: 29 cae en «18-29» y 30 en «30-44». `sort=False` mantiene el orden de los grupos.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P05_3.json")

## <font color='steelblue'>4. Porcentajes sin trampas</font>

Tres errores habituales en prensa:

1. **Confundir la base**: «el 60 % de los que comparten noticias ha creído un bulo» no es lo mismo que «el 60 % de los que han creído un bulo comparte noticias». Di siempre **sobre quién** se calcula el porcentaje.
2. **Porcentajes de grupos pequeños**: el 50 % de 4 personas son 2 personas. Publica también el número de casos.
3. **Puntos frente a porcentaje**: si un medio pasa del 20 % al 25 % de audiencia, sube **5 puntos**, que es un aumento del **25 %**. Lo verás en el cuaderno 14.

In [ ]:
pago = encuesta[encuesta["suscripcion_pago"] == "Sí"]
print("Personas que pagan por noticias:", len(pago))
print("De ellas, con estudios universitarios (%):",
      round((pago["estudios"] == "Universitarios").mean() * 100, 1))

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P05_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Haz la tabla de frecuencias absolutas y relativas (en %) de la provincia de residencia.

<details><summary>Ver solución</summary>

```python
n = encuesta["provincia"].value_counts()
pd.DataFrame({"n": n, "%": (n / n.sum() * 100).round(1)})
```
</details>

### **Ejercicio 2** *(sección 1)*

¿Qué porcentaje de las personas entrevistadas afirma haber creído alguna vez un bulo?

<details><summary>Ver solución</summary>

```python
round((encuesta["ha_creido_bulo"] == "Sí").mean() * 100, 1)
```
</details>

### **Ejercicio 3** *(sección 2)*

Con la tabla acumulada de confianza, ¿qué porcentaje de quienes contestaron da una confianza de 4 o menos?

<details><summary>Ver solución</summary>

```python
conf = encuesta["confianza_medios"].dropna()
round((conf <= 4).mean() * 100, 1)
```
</details>

### **Ejercicio 4** *(sección 3)*

Agrupa `minutos_noticias_dia` en los intervalos [0, 15], (15, 30], (30, 60], (60, 120] y más de 120, y haz su tabla de frecuencias relativas.

<details><summary>Ver solución</summary>

```python
grupos = pd.cut(encuesta["minutos_noticias_dia"], bins=[-1, 15, 30, 60, 120, 1000],
                labels=["15 o menos", "16-30", "31-60", "61-120", "Más de 120"])
(grupos.value_counts(normalize=True, sort=False) * 100).round(1)
```

Empezamos en −1 para que el 0 entre en el primer intervalo.
</details>

### **Ejercicio 5** *(sección 4)*

Escribe un titular y un subtítulo con el dato de los jóvenes (18-29) que se informan sobre todo por redes. El subtítulo debe decir sobre cuántas personas se calcula.

<details><summary>Ver idea clave</summary>

Por ejemplo: «Seis de cada diez jóvenes se informan sobre todo por redes sociales». Subtítulo: «Entre las 202 personas de 18 a 29 años entrevistadas...». Comprueba la cifra con `value_counts(normalize=True)` sobre ese grupo.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- La **frecuencia absoluta** cuenta casos; la **relativa** es su proporción sobre el total (`normalize=True`).
- En variables **ordinales** tiene sentido la frecuencia **acumulada** (`cumsum()`), con las categorías en orden.
- Las variables cuantitativas con muchos valores se agrupan en **intervalos** con `pd.cut()`.
- Di siempre **sobre quién** se calcula un porcentaje y cuántos casos hay detrás.
- Diferencia entre **puntos porcentuales** y **porcentaje de variación**.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Frecuencia absoluta** | Número de veces que aparece un valor. |
| **Frecuencia relativa** | Proporción del total que supone un valor. |
| **Frecuencia acumulada** | Suma de las frecuencias hasta un valor, en variables ordenadas. |
| **Intervalo de clase** | Rango de valores que se agrupan juntos. |
| **Punto porcentual** | Diferencia entre dos porcentajes. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| Las categorías salen desordenadas | `value_counts()` ordena por frecuencia | Usa `reindex(orden)` o `sort_index()` |
| `pd.cut` deja valores como NaN | Algún valor queda fuera de los límites | Amplía el primer o el último límite |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 6 convertirás estas tablas en gráficos y aprenderás a detectar gráficos engañosos.

**Siguiente cuaderno**: [06. Gráficos para variables cualitativas y gráficos que engañan](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P06_graficos_cualitativas.ipynb)